In [2]:
import pandas as pd
import re

In [5]:
df = pd.read_excel("messy_sales_data.xlsx")

In [6]:
print (df.shape)
print (df.head())
print(df.info())

(20, 10)
    ID   Customer Name                     Email         Phone    Age  \
0  1.0       Ali Ahmed             ali@gmail.com  0300-1234567   28.0   
1  2.0     sara khan              SARA@YAHOO.COM   03001234568   35.0   
2  3.0  Muhammad Bilal               bilal@gmail    3219876543    NaN   
3  4.0       Ali Ahmed             ali@gmail.com  0300-1234567   28.0   
4  5.0    Fatima Malik  fatima.malik@hotmail.com  0321-5556789  999.0   

        City Sales Amount        Date Product     Status  
0    Karachi        15000  2024-01-15  Laptop  Completed  
1     lahore       8500.5  15/01/2024   Phone  completed  
2    Karachi       12,500  2024-01-20  Tablet    PENDING  
3    Karachi        15000  2024-01-15  Laptop  Completed  
4  Islamabad         -500  2024/02/05  Laptop  Cancelled  
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ID           

In [7]:
# Woh rows jo bilkul empty hain unhe drop karo
df.dropna(how='all', inplace=True)
print(f"Rows after removing blanks: {len(df)}")


Rows after removing blanks: 19


In [9]:
# String columns se aage peeche ke spaces hatao
str_cols = ['Customer Name', 'Email', 'City', 'Product', 'Status']
for col in str_cols:
    df[col] = df[col].str.strip()

In [10]:
# Name aur City ko Title Case mein
df['Customer Name'] = df['Customer Name'].str.title()
df['City'] = df['City'].str.title()

# Status ko Capitalized
df['Status'] = df['Status'].str.capitalize()

# Email ko lowercase
df['Email'] = df['Email'].str.lower()

In [11]:
email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'

def is_valid_email(email):
    if pd.isna(email):
        return False
    return bool(re.match(email_pattern, str(email)))

df['Email_Valid'] = df['Email'].apply(is_valid_email)
invalid_emails = df[~df['Email_Valid']]
print("Invalid emails:\n", invalid_emails[['Customer Name', 'Email']])

# Invalid emails ko NaN se replace karein
df.loc[~df['Email_Valid'], 'Email'] = None
df.drop(columns='Email_Valid', inplace=True)

Invalid emails:
      Customer Name            Email
2   Muhammad Bilal      bilal@gmail
7      Zara Sheikh  zara@@gmail.com
12     Ahsan Mirza       ahsanmirza


In [12]:
def clean_phone(phone):
    if pd.isna(phone):
        return None
    phone = re.sub(r'\D', '', str(phone))  # sirf digits rakho
    if len(phone) == 11:
        return f"{phone[:4]}-{phone[4:]}"  # 0300-1234567 format
    return None  # invalid length

df['Phone'] = df['Phone'].apply(clean_phone)


In [13]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=False, errors='coerce')
# Ab sab dates ek format mein hongi, invalid dates NaT ho jaengi
print(df['Date'].isna().sum(), "invalid dates mili")

5 invalid dates mili


In [14]:
# String mein commas hata ke number banao
df['Sales Amount'] = df['Sales Amount'].astype(str).str.replace(',', '', regex=False)
df['Sales Amount'] = pd.to_numeric(df['Sales Amount'], errors='coerce')

# Negative values ko 0 ya NaN se replace karein
df.loc[df['Sales Amount'] < 0, 'Sales Amount'] = None

In [15]:
# Age 0-100 ke beech honi chahiye
df.loc[(df['Age'] > 100) | (df['Age'] < 0), 'Age'] = None